<div align="center">

# 🤖 RLAIF — Полный конвейер обучения с подкреплением
### Reinforcement Learning from AI Feedback · Русский язык · GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

</div>

---

## Что мы построим

```
┌─────────────────────────────────────────────────────────────┐
│  Шаг 1. Gemini генерирует датасет инструкций на русском     │
│  Шаг 2. SFT — дообучаем маленькую модель на датасете        │
│  Шаг 3. GRPO — RL-обучение с наградой от Gemini-судьи       │
│  Шаг 4. Сравниваем SFT vs GRPO                              │
└─────────────────────────────────────────────────────────────┘
```

**Ключевые идеи:**
- **SFT** учит модель имитировать хорошие ответы
- **GRPO** учит модель *искать* лучший ответ через пробы и ошибки
- **AI-судья** (Gemini) заменяет дорогую ручную разметку

**Модели:**
| Роль | Модель |
|------|--------|
| Учитель / Судья | `gemini-2.5-flash` (API) |
| Студент (обучаем) | `Qwen/Qwen2.5-0.5B-Instruct` (~500M параметров) |


---
## ⚙️ Установка и настройка

> **Перед запуском:** выберите GPU-ускоритель  
> `Среда выполнения → Сменить среду → T4 GPU`

In [ ]:
# Устанавливаем все необходимые пакеты
!pip install -q \
    transformers \
    trl \
    peft \
    datasets \
    accelerate \
    bitsandbytes

print("✅ Пакеты установлены")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
✅ Пакеты установлены


In [ ]:
import os, json, random, re, getpass
import torch
import numpy as np

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig, GRPOTrainer, GRPOConfig

# Воспроизводимость
random.seed(42); np.random.seed(42); torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Устройство: cuda
GPU: Tesla T4
VRAM: 15.6 GB


---
## 📚 Шаг 1 — Генерация датасета инструкций

Gemini выступает **учителем**: он создаёт задания и эталонные ответы.

**Тема датасета:** объяснение бытовых явлений простым языком (стиль «объясни пятилетнему ребёнку»)

Формат каждого примера:
```
instruction: "Объясни простыми словами, что такое электричество"
response:    "Представь себе воду в трубе..."
```

In [ ]:
# @title Шаг 1 — Генерация датасета инструкций
from google.colab import ai
import json, re, random

TOPICS = [
    "электричество", "гравитация", "фотосинтез", "иммунитет",
    "интернет", "звук", "свет", "память человека",
]

N_SAMPLES = 20
dataset_raw = []

print(f"Генерируем {N_SAMPLES} примеров...\n")

for i in range(N_SAMPLES):
    topic = TOPICS[i % len(TOPICS)]

    prompt = f"""Напиши ОДНО задание и ответ на русском языке.
Тема: {topic}
Стиль: объяснение для школьника 10 лет, без сложных терминов.

Формат — строго JSON, без лишнего текста:
{{"instruction": "...", "response": "..."}}

Инструкция начинается с глагола (Объясни / Расскажи / Опиши).
Ответ — 3-5 предложений с конкретным примером."""

    raw = ai.generate_text(prompt, model_name="google/gemini-2.5-flash-lite")

    try:
        clean = re.sub(r"```(?:json)?|```", "", raw).strip()
        example = json.loads(clean)
        if "instruction" in example and "response" in example:
            dataset_raw.append(example)
            print(f"[{i+1}/{N_SAMPLES}] ✔ {topic}: {example['instruction'][:50]}...")
    except Exception:
        print(f"[{i+1}/{N_SAMPLES}] ✗ Пропускаем (некорректный JSON)")

print(f"\n✅ Собрано примеров: {len(dataset_raw)}")
print("\nПример из датасета:")
ex = random.choice(dataset_raw)
print(f"  Вопрос: {ex['instruction']}")
print(f"  Ответ:  {ex['response'][:100]}...")

Генерируем 20 примеров...

[1/20] ✔ электричество: Объясни, как лампочка загорается, когда мы включае...
[2/20] ✔ гравитация: Объясни, почему яблоко падает вниз, а не летит вве...
[3/20] ✔ фотосинтез: Объясни, как растения 'едят' и дышат с помощью сол...
[4/20] ✔ иммунитет: Объясни, как работает наш внутренний телохранитель...
[5/20] ✔ интернет: Объясни, как работает интернет, чтобы я мог посмот...
[6/20] ✔ звук: Объясни, как звук путешествует от одной вещи к дру...
[7/20] ✔ свет: Объясни, как свет помогает нам видеть....
[8/20] ✔ память человека: Объясни, как работает наша память, как будто она —...
[9/20] ✔ электричество: Объясни, что такое электрический ток простыми слов...
[10/20] ✔ гравитация: Объясни, почему яблоки падают на землю, а не улета...
[11/20] ✔ фотосинтез: Объясни, как растения "едят" и "дышат" с помощью с...
[12/20] ✔ иммунитет: Объясни, как работает наша внутренняя «армия» (имм...
[13/20] ✔ интернет: Объясни, как работает интернет, чтобы я мог смотре...
[14/20] ✔ зву

In [ ]:
from datasets import Dataset

def to_messages(ex: dict) -> dict:
    return {
        "messages": [
            {"role": "system",    "content": "Ты — полезный помощник, объясняющий сложные вещи простым языком."},
            {"role": "user",      "content": ex["instruction"]},
            {"role": "assistant", "content": ex["response"]},
        ],
        # Сохраняем prompt отдельно — понадобится для GRPO позже
        "prompt": ex["instruction"],
    }

formatted = [to_messages(ex) for ex in dataset_raw]
random.shuffle(formatted)

split = int(len(formatted) * 0.8)

# SFTTrainer должен видеть ТОЛЬКО колонку messages
train_data = Dataset.from_list(formatted[:split]).select_columns(["messages"])
test_data  = Dataset.from_list(formatted[split:]).select_columns(["messages"])

# Отдельно храним промпты для GRPO
grpo_prompts = [ex["prompt"] for ex in formatted]

print(f"Train: {len(train_data)} | Test: {len(test_data)}")
print(f"Колонки: {train_data.column_names}")  # Должно быть только ['messages']

Train: 16 | Test: 4
Колонки: ['messages']


---
## 🎓 Шаг 2 — SFT: Supervised Fine-Tuning

**Идея SFT:** показать модели хорошие примеры и попросить их имитировать.

**Что такое LoRA?**
```
Обычное обновление:  W_new = W + ΔW      (ΔW — огромная матрица)
LoRA:               W_new = W + A × B   (A, B — маленькие матрицы)

rank=16 → обучаем <1% параметров вместо 100%
```

**QLoRA = LoRA + квантизация 4-бит** → модель занимает ~800 МБ вместо 2 ГБ

In [ ]:
# Загружаем базовую модель в 4-битном формате (QLoRA)
STUDENT_MODEL="Qwen/Qwen3-0.6B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Загружаем модель: {STUDENT_MODEL}")

base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

params_m = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"✅ Модель загружена | Параметры: {params_m:.0f}M")

Загружаем модель: Qwen/Qwen3-0.6B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Модель загружена | Параметры: 531M


In [ ]:
# Добавляем LoRA-адаптеры к модели

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,           # Ранг: размер матриц A и B
    lora_alpha=32,  # Масштабирование: alpha/r = 2.0
    lora_dropout=0.05,
    # Применяем LoRA к слоям внимания и FFN
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

sft_model = get_peft_model(base_model, lora_config)
sft_model.print_trainable_parameters()
# Ожидаем: ~1-2% обучаемых параметров

trainable params: 10,092,544 || all params: 761,724,928 || trainable%: 1.3250


In [ ]:
# SFT-обучение

SFT_DIR = "/content/sft_model"

sft_trainer = SFTTrainer(
    model=sft_model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=test_data,
    args=SFTConfig(
        output_dir=SFT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        optim="paged_adamw_8bit",
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
    ),
)

print("Запускаем SFT-обучение...")
sft_trainer.train()
sft_trainer.save_model(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)
print(f"✅ SFT-модель сохранена в {SFT_DIR}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Запускаем SFT-обучение...


Epoch,Training Loss,Validation Loss
1,No log,2.528867
2,No log,2.039041
3,2.421087,1.960838


✅ SFT-модель сохранена в /content/sft_model


In [ ]:
# Вспомогательная функция генерации ответа

def generate(model, prompt: str, max_new_tokens: int = 200) -> str:
    """Генерирует ответ модели на заданный промпт."""
    messages = [
        {"role": "system",  "content": "Ты — полезный помощник, объясняющий сложные вещи простым языком."},
        {"role": "user",    "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Тест SFT-модели
sft_eval_model = PeftModel.from_pretrained(base_model, SFT_DIR, is_trainable=False)
sft_eval_model.eval()

test_prompt = "Объясни простыми словами, что такое гравитация"
print("📝 Вопрос:", test_prompt)
print("-" * 60)
print("🤖 SFT-ответ:")
print(generate(sft_eval_model, test_prompt))

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


📝 Вопрос: Объясни простыми словами, что такое гравитация
------------------------------------------------------------
🤖 SFT-ответ:
<think>

</think>

Гравитация – это такая сила, которая действует между всеми телами. Например, если ты в комнате и молчал, а другому человеку прикоснешься, то его ощущения будут появляться потому, что мы несли в нём силу. Это как будто ты смотрел на звезды из-за неба, и ты чувствуешь её за счёт звёзд.


---
## 🎯 Шаг 3 — GRPO: Group Relative Policy Optimization

### Чем GRPO лучше DPO?

| | DPO | GRPO |
|---|---|---|
| Данные | Готовые пары (chosen, rejected) | Только промпты |
| Обучение | Один проход по датасету | Онлайн: генерация + оценка |
| Сигнал | Бинарный (лучше/хуже) | Числовая награда |
| Гибкость | Низкая | Высокая |

### Как работает GRPO?

```
Для каждого промпта:
  1. Генерируем G ответов  →  [y₁, y₂, ..., yG]
  2. Оцениваем каждый      →  [r₁, r₂, ..., rG]
  3. Нормализуем наград    →  ã_i = (r_i - mean(r)) / std(r)
  4. Обновляем модель:     →  хорошие ответы вероятнее, плохие — реже
```

**Loss GRPO** (упрощённо):
$$\mathcal{L} = -\mathbb{E}\left[\tilde{A}_i \cdot \log\pi_\theta(y_i|x)\right] + \beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$$

- $\tilde{A}_i$ — нормализованная награда (advantage)
- KL-член не даёт модели «улетать» далеко от SFT-базы

In [ ]:
# ── Функция награды через Gemini-судью ────────────────────────────────────────
#
# GRPOTrainer вызывает reward_fn(prompts, completions) → List[float]
# Мы просим Gemini оценить каждый ответ по шкале 1-10

JUDGE_PROMPT = """\
Оцени качество ответа на русском языке по шкале от 1 до 10.

Критерии оценки:
- Точность (правильность информации)
- Простота (понятно ли школьнику)
- Наличие конкретного примера или аналогии

Вопрос: {prompt}
Ответ: {response}

Ответь ТОЛЬКО одним целым числом от 1 до 10. Никакого пояснения."""


def gemini_reward(prompts: list[str], completions: list[str], **kwargs) -> list[float]:
    """
    Функция награды для GRPO.
    GRPOTrainer передаёт:
      prompts     — список промптов (батч × num_generations)
      completions — список ответов модели для каждого промпта
    Возвращает список float-наград.
    """
    rewards = []
    for prompt, completion in zip(prompts, completions):
        judge_input = JUDGE_PROMPT.format(
            prompt=prompt,
            response=completion,
        )
        try:
            raw = ask_gemini(judge_input, temperature=0.0)  # Детерминированный судья
            score = float(re.search(r"\d+", raw).group())
            score = max(1.0, min(10.0, score))  # Зажимаем в [1, 10]
        except Exception:
            score = 5.0  # Нейтральная оценка при ошибке
        rewards.append(score)
    return rewards


# Тест функции награды
test_scores = gemini_reward(
    prompts=["Объясни, что такое фотосинтез", "Объясни, что такое фотосинтез"],
    completions=[
        "Фотосинтез — это процесс, при котором растения с помощью света создают сахар из воды и углекислого газа. Представь, что лист — это маленькая солнечная батарейка и кухня одновременно.",
        "Растения едят свет.",
    ]
)
print("Тест функции награды:")
print(f"  Хороший ответ → оценка: {test_scores[0]:.0f}/10")
print(f"  Плохой ответ  → оценка: {test_scores[1]:.0f}/10")

Тест функции награды:
  Хороший ответ → оценка: 5/10
  Плохой ответ  → оценка: 5/10


In [ ]:
# Датасет для GRPO — нужна только колонка 'prompt'
# GRPO сам генерирует ответы в процессе обучения

grpo_dataset = Dataset.from_dict({
    "prompt": [ex["prompt"] for ex in formatted]
})

print(f"GRPO датасет: {len(grpo_dataset)} промптов")
print("Примеры промптов:")
for p in grpo_dataset["prompt"][:3]:
    print(f"  • {p}")

GRPO датасет: 20 промптов
Примеры промптов:
  • Объясни, почему тень появляется, когда свет встречает препятствие.
  • Объясни, что такое электрический ток простыми словами, как будто это река.
  • Объясни, как работает наш внутренний телохранитель – иммунитет.


In [ ]:
# Загружаем SFT-модель как стартовую точку для GRPO
# GRPO будет обучать эту модель, используя SFT-веса как reference (замороженную копию)

grpo_base = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
grpo_base.config.use_cache = False

# Инициализируем LoRA из SFT-чекпоинта
grpo_model = PeftModel.from_pretrained(grpo_base, SFT_DIR, is_trainable=True)

print("✅ GRPO-модель готова (инициализирована из SFT-весов)")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ GRPO-модель готова (инициализирована из SFT-весов)


In [ ]:
# @title Шаг 3 — GRPO обучение

GRPO_DIR = "/content/grpo_model"

grpo_trainer = GRPOTrainer(
    model=grpo_model,
    processing_class=tokenizer,
    reward_funcs=[gemini_reward],
    train_dataset=Dataset.from_dict({"prompt": grpo_prompts}),
    args=GRPOConfig(
        output_dir=GRPO_DIR,
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=4,
        beta=0.04,
        learning_rate=5e-6,
        optim="paged_adamw_8bit",
        logging_steps=1,
        save_strategy="epoch",
        bf16=True,
        report_to="none",
    ),
)

print("Запускаем GRPO-обучение...")
print(f"  Ответов на промпт (G): 4")
print(f"  KL-коэффициент  (β): 0.04\n")
grpo_trainer.train()
grpo_trainer.save_model(GRPO_DIR)
tokenizer.save_pretrained(GRPO_DIR)
print(f"\n✅ GRPO-модель сохранена в {GRPO_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Запускаем GRPO-обучение...
  Ответов на промпт (G): 4
  KL-коэффициент  (β): 0.04



Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
1,0.000053
2,0.000054
3,0.000047
4,0.000041
5,0.000054
6,0.000049
7,0.000049
8,0.000057
9,0.000052
10,0.000047


Step,Training Loss
1,0.000053
2,0.000054
3,0.000047
4,0.000041
5,0.000054
6,0.000049
7,0.000049
8,0.000057
9,0.000052
10,0.000047


---
## 📊 Шаг 4 — Сравнение: SFT vs GRPO

Задаём одинаковые вопросы обеим моделям, судья оценивает ответы.

In [ ]:
# Загружаем GRPO-модель для инференса
grpo_eval_model = PeftModel.from_pretrained(grpo_base, GRPO_DIR, is_trainable=False)
grpo_eval_model.eval()

# Тестовые вопросы (не из обучающей выборки)
TEST_QUESTIONS = [
    "Объясни простыми словами, как работает самолёт",
    "Расскажи, почему небо синее",
    "Объясни, что такое чёрная дыра",
    "Расскажи, как работает мозг",
]

results = []

print(f"{'Вопрос':<45} {'SFT':>5} {'GRPO':>5} {'Победитель':>12}")
print("-" * 72)

for q in TEST_QUESTIONS:
    sft_ans  = generate(sft_eval_model,  q)
    grpo_ans = generate(grpo_eval_model, q)

    # Оцениваем оба ответа
    scores = gemini_reward(
        prompts=[q, q],
        completions=[sft_ans, grpo_ans],
    )
    sft_score, grpo_score = scores[0], scores[1]
    winner = "GRPO 🏆" if grpo_score > sft_score else ("Ничья" if grpo_score == sft_score else "SFT")

    results.append({"question": q, "sft": sft_ans, "grpo": grpo_ans,
                    "sft_score": sft_score, "grpo_score": grpo_score})

    short = q[:42] + "..." if len(q) > 45 else q
    print(f"{short:<45} {sft_score:>5.1f} {grpo_score:>5.1f} {winner:>12}")

print("-" * 72)
avg_sft  = np.mean([r["sft_score"]  for r in results])
avg_grpo = np.mean([r["grpo_score"] for r in results])
print(f"{'СРЕДНЯЯ ОЦЕНКА':<45} {avg_sft:>5.1f} {avg_grpo:>5.1f}")
print(f"\n📈 Улучшение GRPO: {avg_grpo - avg_sft:+.1f} баллов")

In [ ]:
# Детальное сравнение лучшего примера
best = max(results, key=lambda r: r["grpo_score"] - r["sft_score"])

print("=" * 65)
print("НАИБОЛЬШЕЕ УЛУЧШЕНИЕ")
print("=" * 65)
print(f"\n❓ Вопрос: {best['question']}\n")
print(f"📘 SFT-ответ  (оценка {best['sft_score']:.0f}/10):")
print(best["sft"])
print(f"\n📗 GRPO-ответ (оценка {best['grpo_score']:.0f}/10):")
print(best["grpo"])
print("=" * 65)

In [ ]:
# Визуализация результатов
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.family"] = "DejaVu Sans"  # Поддержка кириллицы

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("RLAIF: SFT vs GRPO", fontsize=14, fontweight="bold")

# График 1: оценки по вопросам
x = np.arange(len(results))
ax1.bar(x - 0.2, [r["sft_score"]  for r in results], 0.4, label="SFT",  color="#4A90D9")
ax1.bar(x + 0.2, [r["grpo_score"] for r in results], 0.4, label="GRPO", color="#E67E22")
ax1.axhline(avg_sft,  color="#4A90D9", linestyle="--", alpha=0.7)
ax1.axhline(avg_grpo, color="#E67E22", linestyle="--", alpha=0.7)
ax1.set_xticks(x)
ax1.set_xticklabels([f"Q{i+1}" for i in range(len(results))])
ax1.set_ylim(0, 11)
ax1.set_ylabel("Оценка судьи (/ 10)")
ax1.set_title("Оценки по вопросам")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# График 2: среднее
ax2.bar(["SFT", "GRPO"], [avg_sft, avg_grpo],
        color=["#4A90D9", "#E67E22"], width=0.4)
ax2.set_ylim(0, 11)
ax2.set_ylabel("Средняя оценка")
ax2.set_title("Итог")
for i, v in enumerate([avg_sft, avg_grpo]):
    ax2.text(i, v + 0.2, f"{v:.1f}", ha="center", fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/content/sft_vs_grpo.png", dpi=150)
plt.show()
print("📊 График сохранён")

---
## 🏁 Итоги

### Что мы сделали

| Шаг | Техника | Ключевая идея |
|-----|---------|---------------|
| 1 | **Генерация датасета** | Gemini = учитель, создаёт обучающие примеры |
| 2 | **SFT + QLoRA** | Имитация хороших ответов, обучаем <1% параметров |
| 3 | **GRPO** | RL-поиск лучших ответов, наград даёт Gemini-судья |
| 4 | **Оценка** | AI-судья сравнивает модели объективно |

### Как GRPO улучшает модель
```
SFT:  модель запоминает → "как выглядит хороший ответ"
GRPO: модель учится    → "как найти ещё лучший ответ"
```

### Параметры для экспериментов

| Параметр | Что меняет | Попробуй |
|----------|------------|----------|
| `num_generations` (G) | Размер группы для сравнения | 4 → 8 |
| `beta` | Близость к SFT-базе | 0.04 → 0.1 |
| `learning_rate` | Скорость RL-обновлений | 5e-6 → 1e-5 |
| `N_SAMPLES` | Размер датасета | 30 → 200 |
